# Whisper Large-v3 Levantine 1x1x1 Plug-and-Play Run

This notebook reuses the existing Levantine Whisper large-v3 manifests from this repo, materializes a safe local `1 train / 1 validation / 1 test` dataset, trains the plug-and-play Whisper backend, and evaluates the held-out test sample.

In [ ]:
from pathlib import Path
import subprocess
import sys

PROJECT_DIR = Path.cwd()
PREP_SCRIPT = PROJECT_DIR / "prepare_levantine_whisper_large_v3_1x1x1.py"
TRAINER = PROJECT_DIR / "asr_universal_trainer.py"
EVAL_SCRIPT = PROJECT_DIR / "evaluate_whisper_checkpoint.py"
CONFIG_PATH = PROJECT_DIR / "whisper_large_v3_levantine_1x1x1.yaml"
WORK_DIR = PROJECT_DIR / "runs" / "whisper_large_v3_levantine_1x1x1"
DATASET_DIR = PROJECT_DIR / "datasets" / "whisper_large_v3_levantine_1x1x1"

print("Project dir:", PROJECT_DIR)
print("Prep script:", PREP_SCRIPT)
print("Trainer:", TRAINER)
print("Eval script:", EVAL_SCRIPT)
print("Config:", CONFIG_PATH)
print("Work dir:", WORK_DIR)


## 1. Materialize the 1x1x1 dataset

This step keeps the same source manifests and audio decoding logic as the existing repo run, but writes local WAV files so the generic trainer can consume them.

In [ ]:
cmd = [sys.executable, str(PREP_SCRIPT)]
print("Running:", " ".join(cmd))
result = subprocess.run(cmd, cwd=PROJECT_DIR, text=True, capture_output=True)
print(result.stdout)
if result.stderr:
    print(result.stderr)
assert result.returncode == 0, result.returncode


In [ ]:
for rel in ["dataset_summary.json", "train.jsonl", "validation.jsonl", "test.jsonl"]:
    path = DATASET_DIR / rel
    print(f"\n--- {path} ---")
    if path.exists():
        print(path.read_text(encoding="utf-8")[:4000])
    else:
        print("missing")


## 2. Show the run config

In [ ]:
print(CONFIG_PATH.read_text(encoding="utf-8"))


## 3. Prepare the trainer dataset

In [ ]:
cmd = [sys.executable, str(TRAINER), "--config", str(CONFIG_PATH), "--stage", "prepare"]
print("Running:", " ".join(cmd))
result = subprocess.run(cmd, cwd=PROJECT_DIR, text=True, capture_output=True)
print(result.stdout)
if result.stderr:
    print(result.stderr)
assert result.returncode == 0, result.returncode


## 4. Train Whisper large-v3 on the 1-sample train split

In [ ]:
cmd = [sys.executable, "-u", str(TRAINER), "--config", str(CONFIG_PATH), "--stage", "train"]
print("Running:", " ".join(cmd))
process = subprocess.Popen(cmd, cwd=PROJECT_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in process.stdout:
    print(line, end="")
rc = process.wait()
assert rc == 0, rc


## 5. Evaluate the best checkpoint on the 1-sample test split

In [ ]:
cmd = [sys.executable, "-u", str(EVAL_SCRIPT), "--config", str(CONFIG_PATH), "--split", "test"]
print("Running:", " ".join(cmd))
process = subprocess.Popen(cmd, cwd=PROJECT_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in process.stdout:
    print(line, end="")
rc = process.wait()
assert rc == 0, rc


## 6. Inspect logs and outputs

In [ ]:
for rel in [
    "run.log",
    "metrics.csv",
    "run_result.json",
    "prepared/stats.json",
    "whisper_test_metrics.json",
    "whisper_test_predictions.json",
]:
    path = WORK_DIR / rel
    print(f"\n--- {path} ---")
    if path.exists():
        print(path.read_text(encoding="utf-8")[-5000:])
    else:
        print("missing")
